# Gemini 4 Argon: The Economics of a Million-Token Output

Companion notebook for the 2026-10-01 post. We work through three things with real numbers from the Sep 30, 2026 launch coverage:

1. **Pricing math** — what a realistic workload costs across Argon (intro vs standard), Claude Opus 5.5, Claude Sonnet 5.5, and GPT-6.1 Sol.
2. **Output-budget economics** — what 1M output tokens actually means, and why output tokens dominate the bill.
3. **Benchmark margins** — how big Argon's reported leads really are, computed as relative improvements over the best rival.

*All figures are vendor/third-party-reported as of Sep 30 / Oct 1, 2026 and had not been independently re-run at launch.*

In [ ]:
# $ per 1M tokens: (input, cached_input, output). Cached read assumed 90% of input.
PRICING = {
    "Gemini 4 Argon (intro)":    {"in": 2.00, "cached": 0.10, "out": 10.00},
    "Gemini 4 Argon (standard)": {"in": 4.00, "cached": 0.20, "out": 20.00},  # cached: ~95% off input
    "Claude Opus 5.5":           {"in": 4.00, "cached": 0.20, "out": 20.00},
    "Claude Sonnet 5.5":         {"in": 2.00, "cached": 0.20, "out": 10.00},
    "GPT-6.1 Sol":               {"in": 2.00, "cached": 0.10, "out": 10.00},
}

def workload_cost(name, input_tok, output_tok, cache_hit_rate=0.9):
    p = PRICING[name]
    in_cost = (input_tok / 1e6) * (cache_hit_rate * p["cached"] + (1 - cache_hit_rate) * p["in"])
    out_cost = (output_tok / 1e6) * p["out"]
    return in_cost + out_cost

# Scenario: an agentic coding session — 2M input tokens (90% cache hits), 500K output tokens
scenario = dict(input_tok=2_000_000, output_tok=500_000)
for name in PRICING:
    print(f"{name:28s} ${workload_cost(name, **scenario):7.2f}")

c_intro = workload_cost("Gemini 4 Argon (intro)", **scenario)
c_opus = workload_cost("Claude Opus 5.5", **scenario)
assert abs(c_intro - (2*0.29 + 5.0)) < 1e-9  # 2M in @ ~$0.29/M blended + 500K out @ $10/M
assert c_intro < c_opus / 1.9, "intro Argon should cost under half of Opus 5.5 here"
print(f"\nArgon intro is {c_opus / c_intro:.2f}x cheaper than Opus 5.5 on this workload")

## 2. What does a 1M-token output budget actually mean?

Rule of thumb: 1 token ≈ 0.75 words; a book page ≈ 250 words; a line of code ≈ 10 tokens.

In [ ]:
def token_equivalents(tok):
    words = tok * 0.75
    return {"words": words, "pages": words / 250, "code_lines": tok / 10}

for label, tok in [("Gemini 3.x cap", 64_000), ("Argon cap", 1_000_000)]:
    e = token_equivalents(tok)
    print(f"{label:15s}: {e['pages']:7,.0f} pages | {e['code_lines']:8,.0f} lines of code")

e64, e1m = token_equivalents(64_000), token_equivalents(1_000_000)
assert abs(e1m["pages"] / e64["pages"] - 1_000_000 / 64_000) < 1e-9
assert 2900 < e1m["pages"] < 3100, "~3,000 pages per maxed-out call"

# The bill for actually USING the full output budget, one call, intro pricing:
full_call_out = (1_000_000 / 1e6) * PRICING["Gemini 4 Argon (intro)"]["out"]
print(f"\nOne maxed-out 1M-output call (intro price): ${full_call_out:.2f} of output tokens")
print(f"Same call at standard $20/M output:          ${(1_000_000/1e6)*20:.2f}")
assert full_call_out == 10.0

## 3. Benchmark margins: relative improvement over the best rival

Google-reported Argon scores vs the best reported rival score on each benchmark.

In [ ]:
# (benchmark, argon, best_rival, rival_name)
BENCH = [
    ("DeepSWE v1.1",        77.9, 74.2, "Claude Opus 5.5"),
    ("AutomationBench",      51.3, 42.5, "Claude Opus 5.5"),
    ("Harvey Legal Agent",   19.6,  5.4, "GPT-6 Astra"),
    ("Vals Finance Agent",   65.4, 58.4, "next best (~7pts back)"),  # ~7 points clear per coverage
]

print(f"{'benchmark':20s} {'argon':>6s} {'rival':>6s} {'abs Δ':>7s} {'rel gain':>9s}")
for name, a, r, rn in BENCH:
    abs_d, rel = a - r, (a - r) / r
    print(f"{name:20s} {a:6.1f} {r:6.1f} {abs_d:+6.1f}pp {rel:+8.1%}")

# Sanity checks on the headline claims
harvey = [b for b in BENCH if b[0] == "Harvey Legal Agent"][0]
assert harvey[1] / harvey[2] > 3.6, "Harvey: Argon ~3.6x+ the best rival"
deep = [b for b in BENCH if b[0] == "DeepSWE v1.1"][0]
assert 0.04 < (deep[1] - deep[2]) / deep[2] < 0.06, "DeepSWE margin is ~5% relative, not a blowout"
print("\nTakeaway: the biggest margins are in professional-agent work (Harvey +263%),")
print("not in raw coding — DeepSWE is a +3.7pp / +5.0% relative edge.")

## Wrap-up

- On a realistic agentic workload (2M input @ 90% cache hits, 500K output), Argon's **intro** pricing runs at roughly **half** the cost of Claude Opus 5.5 — matching the launch claim.
- A maxed-out 1M-output call costs **$10 in output tokens** at intro pricing ($20 standard) — output is where the budget goes.
- The benchmark story is really a **professional-work** story: +263% relative on Harvey Legal vs +5% on DeepSWE.
- Caveats from the post still apply: vendor-reported numbers, mixed methodologies, and the disputed internal-dissent report. Re-run before you bet a roadmap on it.